# Active learning with a committee

This notebook implements the selection procedure we used to grow the labelled set from 723 to about 1,900 crops. Every round, a committee of five models grades all unlabelled crops, and the 100 crops they disagree on most usefully are sent for labelling. The method, with references, is described below and in `docs/method.md`.

The notebook runs in two modes:

- **Simulation** (`MODE = 'simulate'`): the public dataset already contains the labels of every crop we selected, and of the rest of the pool. The notebook hides them and reveals a crop's grade only when the committee selects it. This reproduces the loop without any labelling, and it can be compared with random selection (`ARM = 'random'`).
- **Your own data** (`MODE = 'label'`): selected crops are exported, or uploaded to your own Roboflow project, for you to grade; the labels are then read back in.

## How the 100 crops are chosen

**Query-by-committee.** Several models trained on the same labels are asked to grade every unlabelled crop. Crops on which they disagree are the ones whose label would tell the models the most [Seung et al. 1992; Freund et al. 1997]. Ensembles of deep networks give better uncertainty estimates than a single network [Lakshminarayanan et al. 2017], and they work well for selecting images to label [Beluch et al. 2018].

**The committee.** A committee only helps if its members make different mistakes. Models differ most when they were pretrained on different data or with different objectives, less when only the architecture differs, and least when only training settings differ [Gontijo-Lopes et al. 2022; Mustafa et al. 2020; Zaidi et al. 2021]. The five members therefore cover three pretraining objectives, two architecture families and two training regimes:

| Member | Pretraining | Training here |
|---|---|---|
| DINOv3 ViT-S/16, 448 px | self-supervised [Siméoni et al. 2025] | last 4 blocks fine-tuned |
| DINOv3 ViT-S/16, 224 px | self-supervised | frozen, head only |
| DINOv3 ConvNeXt-Tiny, 448 px | distilled from the DINOv3 teacher | last 2 stages fine-tuned |
| ConvNeXt-Tiny, 224 px | supervised, ImageNet-22k labels | frozen, head only |
| SigLIP 2 B/16, 224 px | image–text pairs | frozen, head only |

All five use the same ordinal head (CORN, [Shi et al. 2023]), which gives P(grade > t) for the five boundaries A|B, B|C, C|D, D|E and E|F. In the study, a frozen DINOv2 ViT-S [Oquab et al. 2023] was the fourth member in rounds 1–2. It was replaced by the supervised ConvNeXt because its errors were too close to those of the DINOv3 members. Section 5 measures how differently the members actually behave [Geirhos et al. 2021; Kuncheva & Whitaker 2003].

**Step 1: disagreement per grade boundary.** For each boundary, the score is the BALD quantity [Houlsby et al. 2011]: the entropy of the committee's average P(grade > t) minus the average entropy of the members. It is large only when the members are individually confident but disagree with each other. A crop that is unclear to everyone (blurry, for instance) scores low, which is what we want, because a label rarely resolves those. The five boundary scores are summed with weights 1, 2, 2, 2, 1. Disagreement at B|C, C|D and D|E, the boundaries that decide grades C and D, counts double, because C and D matter most for rescue work.

**Step 2: typicality.** The score is multiplied by how typical the crop is: its mean cosine similarity to its 20 nearest neighbours in the pool, using frozen DINOv3 features. Otherwise odd one-off crops tend to dominate a batch [Settles 2012; Kee et al. 2018].

**Step 3: uncertain first, then diverse.** The 500 highest-scoring crops are the candidates. Those that the committee grades C or D, or whose strongest disagreement lies at B|C, C|D or D|E, form a C/D group with 70 of the 100 places. The others get 30 places. Within each group the candidates are clustered with k-means into as many clusters as there are places, and the crop nearest each cluster centre is chosen. At most one crop per photo is allowed. This follows the structure of DropQuery [Gupte et al. 2024] and the typicality idea of TypiClust [Hacohen et al. 2022]. Picking only the top-100 most uncertain crops would give many near-identical crops [Kirsch et al. 2019; Ash et al. 2020a; Sener & Savarese 2018].

**Retraining between rounds.** Frozen members are retrained from scratch each round (seconds). Fine-tuned members continue from the previous round's weights with a re-initialised head [Nikishin et al. 2022; Kumar et al. 2022], for at most 10 epochs. Every third round is trained from scratch, because warm-started networks can end up slightly worse than fresh ones [Ash & Adams 2020; Dohare et al. 2024].

In [ ]:
# On Colab: install the package and get the configuration files.
!pip -q install "insarag-triage[roboflow] @ git+https://github.com/leeelyehezkel/insarag-damage-triage"
!git clone -q https://github.com/leeelyehezkel/insarag-damage-triage repo 2>/dev/null || true

In [ ]:
from pathlib import Path
import dataclasses
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from insarag_triage import core as L, active_learning as AL, data as DATA

REPO = Path('repo') if Path('repo').exists() else Path('.')
WORK = Path('work')                 # everything this notebook writes goes here
MODE = 'simulate'                   # 'simulate' or 'label'
ARM = 'committee'                   # 'committee' or 'random' (simulation only: the comparison with random selection)
N_ROUNDS = 3                        # simulated rounds to run in this session (finished rounds are skipped)

BUDGET, FOCUS_SHARE, N_CANDIDATES, MAX_PER_PHOTO = 100, 0.7, 500, 1
SEEDS, PARTIAL_SEEDS = [0, 1, 2], [0]
WARM_START, WARM_EPOCHS, RESET_HEAD, COLD_EVERY = True, 10, True, 3

COMMITTEE = [
    L.Config('v3_s_partial_448', backbone='dinov3_vits16', mode='partial', img_size=448, lr_backbone=2e-5),
    L.Config('v3_s_frozen_224', backbone='dinov3_vits16', mode='frozen', img_size=224),
    L.Config('cnx_in22k_frozen_224', backbone='convnext_tiny_in22k', mode='frozen', img_size=224),
    L.Config('v3_cnxt_partial_448', backbone='dinov3_convnext_tiny', mode='partial', img_size=448, lr_backbone=5e-5),
    L.Config('siglip2_frozen_224', backbone='siglip2_b16', mode='frozen', img_size=224),
]
# For a quick look (minutes per round instead of about an hour), use the three frozen members only:
# COMMITTEE = [c for c in COMMITTEE if c.mode == 'frozen']
EMB_CFG = COMMITTEE[1] if len(COMMITTEE) > 1 else COMMITTEE[0]     # frozen DINOv3 features for typicality and diversity

## 1. Data

The crops are downloaded from the public Roboflow projects listed in `configs/data.yaml`. This needs a free Roboflow account: the notebook asks for your API key. `data/manifests/labels.csv` gives every crop's grade and split. Test 2 (193 random crops, labelled blind) is used to follow progress; it is never selected or trained on.

In [ ]:
dcfg = DATA.load_config(REPO / 'configs' / 'data.yaml')
folders = {s: DATA.download(dcfg, s, WORK / 'roboflow') for s in ['original', 'al_labels', 'test12']}
m = DATA.assemble(REPO / 'data' / 'manifests' / 'labels.csv', folders)
pd.crosstab(m.split, m.label)

In [ ]:
def as_training_frame(train_rows):
    # train = given rows, val = the study's validation set, test = test 2
    parts = [train_rows.assign(split='train'), m[m.split == 'val'], m[m.split == 'test2'].assign(split='test')]
    return pd.concat(parts, ignore_index=True)

STATE = WORK / f'labelled_{MODE}_{ARM}.csv'
if MODE == 'simulate':
    start = m[(m.split == 'train') & (m.subset == 'original')]
    hidden = m[(m.split == 'train') & (m.subset == 'active_learning')].reset_index(drop=True)
    labelled = pd.read_csv(STATE) if STATE.exists() else start.assign(round=0)
    pool = hidden[~hidden.crop_id.isin(labelled.crop_id)].drop(columns=['label', 'y']).reset_index(drop=True)   # grades hidden
    print(f'labelled: {len(labelled)} crops · unlabelled pool (labels hidden): {len(pool)} crops')
else:
    print('MODE = label: put your own labelled crops in `labelled` and your unlabelled crops in `pool` '
          '(DataFrames with columns path, group = photo id; labelled also needs label and y).')

## 2. One round

Run the next cell once per round. It trains the committee on everything labelled so far, scores the pool, and picks the batch. In simulation it then reveals the labels of the picked crops and adds them to the training set.

In [ ]:
rounds_done = int(labelled['round'].max()) if len(labelled) else 0
for _ in range(N_ROUNDS):
    k = rounds_done + 1
    RD = WORK / f'al_{ARM}' / f'r{k}'
    df = as_training_frame(labelled)
    res = AL.train_committee(COMMITTEE, df, SEEDS, RD, warm_start=WARM_START, warm_epochs=WARM_EPOCHS,
                             partial_seeds=PARTIAL_SEEDS, reset_head=RESET_HEAD, cold_every=COLD_EVERY)
    AL.round_table(res, k, int((df.split == 'train').sum())).to_csv(RD / 'round_table.csv', index=False)
    if len(pool) == 0:
        print('pool is empty'); break
    if ARM == 'random':
        sel = pool.groupby('group', group_keys=False).sample(1, random_state=k).sample(min(BUDGET, pool.group.nunique()), random_state=k)
    else:
        scores = AL.score_pool(COMMITTEE, SEEDS, RD, pool)
        emb = L.embed(EMB_CFG, pool['path']).numpy()
        sel = AL.select_batch(pool, scores, emb, budget=BUDGET, focus_share=FOCUS_SHARE, n_candidates=N_CANDIDATES,
                              max_per_photo=MAX_PER_PHOTO, density_power=1.0)
    sel.to_csv(RD / 'selected.csv', index=False)
    if MODE != 'simulate':
        print(f'round {k}: {len(sel)} crops selected → {RD / "selected.csv"}; label them, then add them to `labelled`'); break
    labelled = pd.concat([labelled, hidden[hidden.crop_id.isin(sel.crop_id)].assign(round=k)], ignore_index=True)
    labelled.to_csv(STATE, index=False)
    pool = pool[~pool.crop_id.isin(sel.crop_id)].reset_index(drop=True)
    print(f"round {k}: +{len(sel)} crops, grades {sel.merge(hidden[['crop_id','label']], on='crop_id').label.value_counts().reindex(L.CLASSES, fill_value=0).to_dict()}")
    rounds_done = k

## 3. Labelling your own batch (MODE = 'label')

The selected crops can be uploaded to **your own** Roboflow classification project. Each crop gets the committee's grade pre-filled and two tags (`al_round_<k>`, `pred_<grade>`). The upload is checked afterwards, because Roboflow occasionally drops images silently. After correcting the grades, generate a version without resizing or augmentation, and read it back with `AL.ingest_labels`.

In [ ]:
UPLOAD = False
if UPLOAD and MODE == 'label':
    key = DATA.api_key()
    MY_WORKSPACE, MY_PROJECT = 'your-workspace', 'your-project'
    up = AL.ensure_uploaded(sel, k, key, MY_WORKSPACE, MY_PROJECT)
    up.to_csv(RD / 'upload_check.csv', index=False)
# after labelling:  labelled = AL.ingest_labels(Path('<downloaded version folder>'), labelled, k)

## 4. Progress on test 2

Overall accuracy is the mean of the six per-grade accuracies (balanced accuracy). MAE is the mean distance in grades. A single round moves accuracy by about 0.01–0.02, while the 95% interval on test 2 is about ±0.07, so look at the trend over several rounds.

In [ ]:
tabs = [pd.read_csv(f) for f in sorted((WORK / f'al_{ARM}').glob('r*/round_table.csv'), key=lambda p: int(p.parent.name[1:]))]
held = pd.concat(tabs, ignore_index=True)
display(held[['round', 'member', 'n_train', 'test_acc', 'test_mae'] + [f'test_{c}' for c in L.CLASSES]].round(3))
AL.plot_overall(held, 'test'); plt.show()
AL.plot_per_class(held, 'test', 'acc'); plt.show()

## 5. Is the committee diverse?

For each pair of members, in the latest round: how often they predict different grades (disagreement), and how much their right/wrong patterns agree beyond chance (error consistency, Cohen's kappa). Kappa near 1 means the two members fail on the same crops, so one of them adds little.

In [ ]:
dis, kap, acc = AL.committee_diversity(WORK / f'al_{ARM}', int(held['round'].max()), COMMITTEE, SEEDS)
display(dis); display(kap)

## References

Ash, J. T., & Adams, R. P. (2020). On warm-starting neural network training. arXiv:1910.08475 ·
Ash, J. T., et al. (2020a). Deep batch active learning by diverse, uncertain gradient lower bounds. arXiv:1906.03671 ·
Beluch, W. H., et al. (2018). The power of ensembles for active learning in image classification. CVPR · doi:10.1109/CVPR.2018.00976 ·
Dohare, S., et al. (2024). Loss of plasticity in deep continual learning. Nature 632 · doi:10.1038/s41586-024-07711-7 ·
Freund, Y., et al. (1997). Selective sampling using the query by committee algorithm. Machine Learning 28 · doi:10.1023/A:1007330508534 ·
Geirhos, R., et al. (2021). Partial success in closing the gap between human and machine vision. arXiv:2106.07411 ·
Gontijo-Lopes, R., et al. (2022). No one representation to rule them all. arXiv:2110.12899 ·
Gupte, S. R., et al. (2024). Revisiting active learning in the era of vision foundation models. arXiv:2401.14555 ·
Hacohen, G., et al. (2022). Active learning on a budget. arXiv:2202.02794 ·
Houlsby, N., et al. (2011). Bayesian active learning for classification and preference learning. arXiv:1112.5745 ·
Kee, S., et al. (2018). Query-by-committee improvement with diversity and density in batch active learning. Information Sciences 454–455 · doi:10.1016/j.ins.2018.05.014 ·
Kirsch, A., et al. (2019). BatchBALD. arXiv:1906.08158 ·
Kumar, A., et al. (2022). Fine-tuning can distort pretrained features and underperform out-of-distribution. arXiv:2202.10054 ·
Kuncheva, L. I., & Whitaker, C. J. (2003). Measures of diversity in classifier ensembles. Machine Learning 51 · doi:10.1023/A:1022859003006 ·
Lakshminarayanan, B., et al. (2017). Simple and scalable predictive uncertainty estimation using deep ensembles. arXiv:1612.01474 ·
Mustafa, B., et al. (2020). Deep ensembles for low-data transfer learning. arXiv:2010.06866 ·
Nikishin, E., et al. (2022). The primacy bias in deep reinforcement learning. arXiv:2205.07802 ·
Oquab, M., et al. (2023). DINOv2. arXiv:2304.07193 ·
Sener, O., & Savarese, S. (2018). A core-set approach. arXiv:1708.00489 ·
Settles, B. (2012). Active Learning. Springer · doi:10.1007/978-3-031-01560-1 ·
Seung, H. S., et al. (1992). Query by committee. COLT · doi:10.1145/130385.130417 ·
Shi, X., et al. (2023). Rank-consistent ordinal regression based on conditional probabilities. Pattern Analysis and Applications 26 · doi:10.1007/s10044-023-01181-9 ·
Siméoni, O., et al. (2025). DINOv3. arXiv:2508.10104 ·
Zaidi, S., et al. (2021). Neural ensemble search for uncertainty estimation and dataset shift. arXiv:2006.08573